In [ ]:
# %% Cell 1：参数、路径

from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from IPython.display import display


# =========================
# 路径
# =========================

# 输出目录
OUT_DIR = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/"
    "Fig4_AS_regulatory_network"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)


# rMATS
RMATS_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260705_Fig3/rmats_works/rmats_results"
)


# gene expression h5ad
GENE_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260705_Fig3/anno_h5ads"
)


# mouse RBP reference
RBP_FILE = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "ref_datas/RBP/Mus_musculus.RBP.txt"
)


# =========================
# 样本
# =========================

SAMPLES = [
    "G21-4", "G21-7",
    "G21-1", "G21-8",
    "G21-2", "G21-5",
]

GROUP_MAP = {
    "G21-4": "Young",
    "G21-7": "Young",
    "G21-1": "Aged",
    "G21-8": "Aged",
    "G21-2": "AD",
    "G21-5": "AD",
}


# rMATS中每个comparison两组样本顺序
RMATS_SAMPLE_MAP = {
    "control_vs_old": {
        "group1": ["G21-4", "G21-7"],
        "group2": ["G21-1", "G21-8"],
        "comparison": "Aged_vs_Young",
    },
    "control_vs_AD": {
        "group1": ["G21-4", "G21-7"],
        "group2": ["G21-2", "G21-5"],
        "comparison": "AD_vs_Young",
    },
}


# =========================
# 脑区 / AS类型
# =========================

REGIONS = [
    "CTX_1", "CTX_2", "CTX_3",
    "HIP", "CA1", "CA3", "DG",
    "TH", "HY", "STR", "FT", "VS",
]

AS_TYPES = [
    "SE",
    "MXE",
    "A3SS",
    "A5SS",
    "RI",
]


# =========================
# AS阈值
# =========================

READ_THRESH = 10
AS_FDR = 0.05
DPSI_THRESH = 0.05


# =========================
# 检查
# =========================

print("Output directory :", OUT_DIR)
print("rMATS directory  :", RMATS_DIR.exists())
print("Gene directory   :", GENE_DIR.exists())
print("RBP reference    :", RBP_FILE.exists())

In [ ]:
# %% Cell 2：读取RBP参考表

rbp = pd.read_csv(
    RBP_FILE,
    sep="\t",
    header=None,
    names=[
        "gene_id",
        "gene_name",
        "entrez_id",
        "aliases",
        "description",
        "RBP_class",
    ],
)

# 基础整理
rbp["gene_id"] = rbp["gene_id"].astype(str).str.strip()
rbp["gene_name"] = rbp["gene_name"].astype(str).str.strip()
rbp["RBP_class"] = rbp["RBP_class"].astype(str).str.strip()

rbp = (
    rbp
    .dropna(subset=["gene_id", "gene_name"])
    .drop_duplicates(["gene_id", "gene_name"])
    .reset_index(drop=True)
)

print("RBP entries :", len(rbp))
print("RBP genes   :", rbp["gene_name"].nunique())
print("Ensembl IDs :", rbp["gene_id"].nunique())

display(rbp.head(20))
display(rbp["RBP_class"].value_counts().head(30))

In [ ]:
# %% Cell 3：读取rMATS结果

# 参数
EVENT_COLS = {
    "SE": ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "MXE": ["chr", "strand", "1stExonStart_0base", "1stExonEnd", "2ndExonStart_0base", "2ndExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "A3SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "A5SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "RI": ["chr", "strand", "riExonStart_0base", "riExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
}

def split_num(x):
    return x.astype(str).str.split(",", expand=True).apply(pd.to_numeric, errors="coerce")


# 读取数据
event_tables = []

for rmats_comp, info in RMATS_SAMPLE_MAP.items():
    for region in REGIONS:
        for ast in AS_TYPES:
            file = RMATS_DIR / rmats_comp / region / f"{ast}.MATS.JC.txt"
            df = pd.read_csv(file, sep="\t")

            total1 = split_num(df["IJC_SAMPLE_1"]) + split_num(df["SJC_SAMPLE_1"])
            total2 = split_num(df["IJC_SAMPLE_2"]) + split_num(df["SJC_SAMPLE_2"])
            coverage = (total1 >= READ_THRESH).all(axis=1) & (total2 >= READ_THRESH).all(axis=1)

            df = df.loc[coverage].copy()
            df["dPSI"] = -pd.to_numeric(df["IncLevelDifference"], errors="coerce")
            df["FDR_num"] = pd.to_numeric(df["FDR"], errors="coerce")
            df["region"] = region
            df["comparison"] = info["comparison"]
            df["AS_type"] = ast
            df["gene_name"] = df["geneSymbol"].astype(str).str.strip().str.replace('"', "", regex=False)

            df["EventID"] = ast + "|" + df[["GeneID"] + EVENT_COLS[ast]].astype(str).agg("|".join, axis=1)
            df["AS_sig"] = (df["FDR_num"] <= AS_FDR) & (df["dPSI"].abs() >= DPSI_THRESH)

            event_tables.append(df[[
                "region", "comparison", "GeneID", "gene_name", "EventID", "AS_type",
                "dPSI", "FDR_num", "AS_sig",
                "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
                "IncLevel1", "IncLevel2"
            ]])

rmats_events = pd.concat(event_tables, ignore_index=True)


# 保存
rmats_events.to_csv(OUT_DIR / "01_rmats_testable_events.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("Testable rows :", len(rmats_events))
print("Unique events :", rmats_events["EventID"].nunique())
print("Significant   :", rmats_events["AS_sig"].sum())
print("Sig events    :", rmats_events.loc[rmats_events["AS_sig"], "EventID"].nunique())

display(rmats_events.head())

In [ ]:
# %% Cell 4：展开sample-level PSI

# 处理数据
psi_tables = []

for rmats_comp, info in RMATS_SAMPLE_MAP.items():
    sub = rmats_events[rmats_events["comparison"] == info["comparison"]].reset_index(drop=True)

    for col, samples in [("IncLevel1", info["group1"]), ("IncLevel2", info["group2"])]:
        psi = sub[col].astype(str).str.split(",", expand=True)
        psi.columns = samples

        tmp = pd.concat([
            sub[["region", "comparison", "gene_name", "EventID", "AS_type"]], psi
        ], axis=1)

        tmp = tmp.melt(
            id_vars=["region", "comparison", "gene_name", "EventID", "AS_type"],
            var_name="sample",
            value_name="PSI"
        )

        tmp["PSI"] = pd.to_numeric(tmp["PSI"], errors="coerce")
        psi_tables.append(tmp)

psi_long_raw = pd.concat(psi_tables, ignore_index=True)
psi_long_raw["condition"] = psi_long_raw["sample"].map(GROUP_MAP)


# 检查Young重复值是否一致
check = psi_long_raw.dropna(subset=["PSI"]).groupby(
    ["sample", "region", "EventID"]
)["PSI"].nunique()

print("PSI rows      :", len(psi_long_raw))
print("Valid PSI     :", psi_long_raw["PSI"].notna().sum())
print("PSI conflicts :", (check > 1).sum())

display(psi_long_raw.head())

In [ ]:
# %% Cell 5：整理PSI并提取候选AS events

# 处理数据
psi_long = (
    psi_long_raw
    .drop_duplicates(["sample", "region", "EventID"])
    .dropna(subset=["PSI"])
    .reset_index(drop=True)
)

candidate_events = rmats_events.loc[rmats_events["AS_sig"], "EventID"].unique()
psi_candidate = psi_long[psi_long["EventID"].isin(candidate_events)].copy()


# 保存
psi_long.to_csv(OUT_DIR / "02_sample_region_event_PSI.tsv.gz", sep="\t", index=False, compression="gzip")
psi_candidate.to_csv(OUT_DIR / "03_candidate_AS_event_PSI.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("All PSI rows       :", len(psi_long))
print("Candidate events   :", len(candidate_events))
print("Candidate PSI rows :", len(psi_candidate))
print("Samples            :", psi_candidate["sample"].nunique())
print("Regions            :", psi_candidate["region"].nunique())

In [ ]:
# %% Cell 6：候选AS event覆盖度QC

# 处理数据
event_qc = (
    psi_candidate
    .groupby(["EventID", "gene_name", "AS_type"])
    .agg(
        n_obs=("PSI", "size"),
        n_samples=("sample", "nunique"),
        n_regions=("region", "nunique"),
        n_conditions=("condition", "nunique"),
    )
    .reset_index()
)

# 保存
event_qc.to_csv(OUT_DIR / "04_candidate_AS_event_QC.tsv", sep="\t", index=False)


# 检查
print("Candidate events:", len(event_qc))

display(event_qc[["n_obs", "n_samples", "n_regions", "n_conditions"]].describe())

print("\nn_samples:")
display(event_qc["n_samples"].value_counts().sort_index())

print("\nn_regions:")
display(event_qc["n_regions"].value_counts().sort_index())

In [ ]:
# %% Cell 7：统计完整6样本脑区

# 处理数据
region_qc = (
    psi_candidate
    .groupby(["EventID", "region"])["sample"]
    .nunique()
    .reset_index(name="n_samples_region")
)

complete_regions = (
    region_qc[region_qc["n_samples_region"] == 6]
    .groupby("EventID")["region"]
    .nunique()
    .reset_index(name="n_complete_regions")
)

event_qc = event_qc.merge(complete_regions, on="EventID", how="left")
event_qc["n_complete_regions"] = event_qc["n_complete_regions"].fillna(0).astype(int)


# 检查
display(event_qc["n_complete_regions"].describe())

print("\nComplete regions:")
display(event_qc["n_complete_regions"].value_counts().sort_index())

In [ ]:
# %% Cell 8：筛选用于调控分析的AS events

# 参数
MIN_COMPLETE_REGIONS = 6


# 处理数据
model_events = event_qc.loc[event_qc["n_complete_regions"] >= MIN_COMPLETE_REGIONS, "EventID"]

complete_pairs = region_qc.loc[
    region_qc["n_samples_region"] == 6,
    ["EventID", "region"]
]

psi_model = psi_candidate[
    psi_candidate["EventID"].isin(model_events)
].merge(complete_pairs, on=["EventID", "region"], how="inner")


# 保存
psi_model.to_csv(OUT_DIR / "05_AS_event_PSI_for_regulatory_model.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("Events  :", psi_model["EventID"].nunique())
print("Genes   :", psi_model["gene_name"].nunique())
print("Regions :", psi_model["region"].nunique())
print("Rows    :", len(psi_model))

display(
    psi_model.groupby("EventID")
    .agg(n_obs=("PSI", "size"), n_regions=("region", "nunique"))
    .describe()
)

In [ ]:
# %% Cell 9：计算sample × region基因平均表达

# 参数
REGION_KEY = "Regions"


# 读取数据
expr_tables = []

for sample in SAMPLES:
    adata = sc.read_h5ad(GENE_DIR / f"{sample}_regions.h5ad")
    gene_id = adata.var_names.astype(str)
    gene_name = adata.var["Symbol"].astype(str).values

    for region in REGIONS:
        mask = adata.obs[REGION_KEY].astype(str).eq(region).values
        if mask.sum() == 0:
            continue

        expression = np.asarray(adata.X[mask].mean(axis=0)).ravel()

        tmp = pd.DataFrame({
            "sample": sample,
            "region": region,
            "gene_id": gene_id,
            "gene_name": gene_name,
            "expression": expression,
        })
        expr_tables.append(tmp)

    print(sample, adata.n_obs, adata.n_vars)
    del adata


# 处理数据
expr_long = pd.concat(expr_tables, ignore_index=True)
expr_long["condition"] = expr_long["sample"].map(GROUP_MAP)


# 保存
expr_long.to_csv(OUT_DIR / "06_sample_region_gene_expression.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("\nRows    :", len(expr_long))
print("Samples :", expr_long["sample"].nunique())
print("Regions :", expr_long["region"].nunique())
print("Genes   :", expr_long["gene_id"].nunique())

display(expr_long.head())

In [ ]:
# %% Cell 10：提取RBP表达并检查覆盖度

# 处理数据
rbp_expr = expr_long.merge(
    rbp[["gene_id", "gene_name", "RBP_class"]].rename(columns={"gene_name": "RBP_name"}),
    on="gene_id",
    how="inner"
)

rbp_qc = (
    rbp_expr.groupby(["gene_id", "RBP_name", "RBP_class"])
    .agg(
        n_obs=("expression", "size"),
        n_positive=("expression", lambda x: (x > 0).sum()),
        n_samples=("sample", "nunique"),
        n_regions=("region", "nunique"),
        mean_expression=("expression", "mean"),
    )
    .reset_index()
)


# 保存
rbp_qc.to_csv(OUT_DIR / "07_RBP_expression_QC.tsv", sep="\t", index=False)


# 检查
print("RBP reference genes :", rbp["gene_id"].nunique())
print("Matched RBP genes   :", rbp_expr["gene_id"].nunique())
print("RBP expression rows :", len(rbp_expr))

display(rbp_qc[["n_obs", "n_positive", "n_samples", "n_regions", "mean_expression"]].describe())

print("\nn_samples:")
display(rbp_qc["n_samples"].value_counts().sort_index())

print("\nn_regions:")
display(rbp_qc["n_regions"].value_counts().sort_index())

In [ ]:
# %% Cell 11：筛选用于调控分析的RBP

# 参数
MIN_RBP_POSITIVE = 36


# 处理数据
rbp_qc["std_expression"] = rbp_expr.groupby("gene_id")["expression"].std().reindex(rbp_qc["gene_id"]).values

rbp_keep = rbp_qc[
    (rbp_qc["n_samples"] == 6) &
    (rbp_qc["n_positive"] >= MIN_RBP_POSITIVE) &
    (rbp_qc["std_expression"] > 0)
].copy()

rbp_expr_keep = rbp_expr[rbp_expr["gene_id"].isin(rbp_keep["gene_id"])].copy()


# 保存
rbp_keep.to_csv(OUT_DIR / "08_RBP_for_regulatory_analysis.tsv", sep="\t", index=False)


# 检查
print("Matched RBP :", len(rbp_qc))
print("Kept RBP    :", len(rbp_keep))

display(
    rbp_keep[["n_positive", "mean_expression", "std_expression"]].describe()
)

In [ ]:
# %% Cell 12：构建RBP expression和PSI矩阵

# 参数
MIN_RBP_OBS = 72


# 处理数据
rbp_model = rbp_keep[rbp_keep["n_obs"] >= MIN_RBP_OBS].drop_duplicates("gene_id").copy()

obs_order = pd.MultiIndex.from_product([SAMPLES, REGIONS], names=["sample", "region"])

expr_mat = (
    rbp_expr_keep[rbp_expr_keep["gene_id"].isin(rbp_model["gene_id"])]
    .pivot_table(index=["sample", "region"], columns="gene_id", values="expression", aggfunc="mean")
    .reindex(obs_order)
)

psi_mat = (
    psi_model
    .pivot_table(index=["sample", "region"], columns="EventID", values="PSI", aggfunc="first")
    .reindex(obs_order)
)

rbp_meta = rbp_model[["gene_id", "RBP_name", "RBP_class"]].set_index("gene_id")
event_meta = psi_model[["EventID", "gene_name", "AS_type"]].drop_duplicates("EventID").set_index("EventID")


# 检查
print("RBP matrix :", expr_mat.shape)
print("PSI matrix :", psi_mat.shape)
print("RBP missing:", expr_mat.isna().sum().sum())
print("AS events  :", psi_mat.shape[1])

In [ ]:
# %% Cell 13：RBP × AS event Pearson相关性

from scipy import stats
from statsmodels.stats.multitest import multipletests

# 参数
COR_THRESH = 0.25
FDR_THRESH = 0.05


# 处理数据
rbp_ids = expr_mat.columns.to_numpy()
event_ids = psi_mat.columns.to_numpy()

R = np.full((len(rbp_ids), len(event_ids)), np.nan, dtype=np.float32)
P = np.full(R.shape, np.nan, dtype=np.float32)

event_regions = (
    psi_model.groupby("EventID")["region"]
    .apply(lambda x: "|".join(sorted(x.unique())))
)

event_pos = {event: i for i, event in enumerate(event_ids)}

for pattern, events in event_regions.groupby(event_regions).groups.items():
    regions = pattern.split("|")
    mask = expr_mat.index.get_level_values("region").isin(regions)

    X = expr_mat.loc[mask].to_numpy(float)
    Y = psi_mat.loc[mask, list(events)].to_numpy(float)
    n = mask.sum()

    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)

    den = np.sqrt((X ** 2).sum(axis=0)[:, None] * (Y ** 2).sum(axis=0)[None, :])

    with np.errstate(divide="ignore", invalid="ignore"):
        r = X.T @ Y / den
        t = r * np.sqrt((n - 2) / (1 - r ** 2))
        p = 2 * stats.t.sf(np.abs(t), df=n - 2)

    cols = [event_pos[event] for event in events]
    R[:, cols] = r.astype(np.float32)
    P[:, cols] = p.astype(np.float32)


# FDR
valid = np.isfinite(P)
PADJ = np.full(P.shape, np.nan, dtype=np.float32)
PADJ[valid] = multipletests(P[valid], method="fdr_bh")[1].astype(np.float32)


# 提取显著边
i, j = np.where((PADJ < FDR_THRESH) & (np.abs(R) >= COR_THRESH))

edges = pd.DataFrame({
    "gene_id": rbp_ids[i],
    "EventID": event_ids[j],
    "pearson_r": R[i, j],
    "pvalue": P[i, j],
    "padj": PADJ[i, j],
})

edges = edges.merge(rbp_meta.reset_index(), on="gene_id")
edges = edges.merge(
    event_meta.reset_index().rename(columns={"gene_name": "AS_gene"}),
    on="EventID"
)

edges["direction"] = np.where(edges["pearson_r"] > 0, "Positive", "Negative")
edges["n_obs"] = edges["EventID"].map(psi_model.groupby("EventID").size())


# 保存
edges.to_csv(OUT_DIR / "09_RBP_AS_Pearson_edges.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("Significant edges :", len(edges))
print("RBP regulators    :", edges["gene_id"].nunique())
print("AS events         :", edges["EventID"].nunique())
print("AS genes          :", edges["AS_gene"].nunique())
print("\nDirection:")
display(edges["direction"].value_counts())

print("\nTop RBP:")
display(edges["RBP_name"].value_counts().head(20))

In [ ]:
# %% Cell 14：Spearman相关性二次筛选

# 参数
COR_THRESH = 0.25
FDR_THRESH = 0.05


# 处理数据
RHO = np.full(R.shape, np.nan, dtype=np.float32)
SP = np.full(R.shape, np.nan, dtype=np.float32)

for pattern, events in event_regions.groupby(event_regions).groups.items():
    regions = pattern.split("|")
    mask = expr_mat.index.get_level_values("region").isin(regions)

    X = stats.rankdata(expr_mat.loc[mask].to_numpy(float), axis=0)
    Y = stats.rankdata(psi_mat.loc[mask, list(events)].to_numpy(float), axis=0)
    n = mask.sum()

    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)

    den = np.sqrt((X ** 2).sum(axis=0)[:, None] * (Y ** 2).sum(axis=0)[None, :])

    with np.errstate(divide="ignore", invalid="ignore"):
        rho = X.T @ Y / den
        t = rho * np.sqrt((n - 2) / (1 - rho ** 2))
        p = 2 * stats.t.sf(np.abs(t), df=n - 2)

    cols = [event_pos[event] for event in events]
    RHO[:, cols] = rho.astype(np.float32)
    SP[:, cols] = p.astype(np.float32)


# FDR
valid = np.isfinite(SP)
SPADJ = np.full(SP.shape, np.nan, dtype=np.float32)
SPADJ[valid] = multipletests(SP[valid], method="fdr_bh")[1].astype(np.float32)


# Pearson + Spearman共同筛选
keep = (
    (PADJ < FDR_THRESH) &
    (SPADJ < FDR_THRESH) &
    (np.abs(R) >= COR_THRESH) &
    (np.abs(RHO) >= COR_THRESH) &
    (np.sign(R) == np.sign(RHO))
)

i, j = np.where(keep)

robust_edges = pd.DataFrame({
    "gene_id": rbp_ids[i],
    "EventID": event_ids[j],
    "pearson_r": R[i, j],
    "pearson_padj": PADJ[i, j],
    "spearman_rho": RHO[i, j],
    "spearman_padj": SPADJ[i, j],
})

robust_edges = robust_edges.merge(rbp_meta.reset_index(), on="gene_id")
robust_edges = robust_edges.merge(
    event_meta.reset_index().rename(columns={"gene_name": "AS_gene"}),
    on="EventID"
)

robust_edges["direction"] = np.where(robust_edges["pearson_r"] > 0, "Positive", "Negative")
robust_edges["n_obs"] = robust_edges["EventID"].map(psi_model.groupby("EventID").size())


# 保存
robust_edges.to_csv(
    OUT_DIR / "10_RBP_AS_Pearson_Spearman_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Robust edges   :", len(robust_edges))
print("RBP regulators :", robust_edges["gene_id"].nunique())
print("AS events      :", robust_edges["EventID"].nunique())
print("AS genes       :", robust_edges["AS_gene"].nunique())

display(robust_edges["direction"].value_counts())
display(robust_edges["RBP_name"].value_counts().head(20))

In [ ]:
# %% Cell 15：校正region和condition后的RBP-AS关联

# 参数
ADJ_COR_THRESH = 0.25
FDR_THRESH = 0.05


# 处理数据
PARTIAL_R = np.full(R.shape, np.nan, dtype=np.float32)
PARTIAL_P = np.full(R.shape, np.nan, dtype=np.float32)

obs_info = pd.DataFrame({
    "sample": expr_mat.index.get_level_values("sample"),
    "region": expr_mat.index.get_level_values("region"),
})
obs_info["condition"] = obs_info["sample"].map(GROUP_MAP)

for pattern, events in event_regions.groupby(event_regions).groups.items():
    regions = pattern.split("|")
    mask = obs_info["region"].isin(regions).values

    X = expr_mat.iloc[mask].to_numpy(float)
    Y = psi_mat.loc[mask, list(events)].to_numpy(float)

    cov = pd.get_dummies(
        obs_info.loc[mask, ["region", "condition"]],
        drop_first=True,
        dtype=float
    )
    C = np.column_stack([np.ones(mask.sum()), cov.to_numpy()])
    Q, _ = np.linalg.qr(C)

    X = X - Q @ (Q.T @ X)
    Y = Y - Q @ (Q.T @ Y)

    den = np.sqrt((X ** 2).sum(axis=0)[:, None] * (Y ** 2).sum(axis=0)[None, :])

    with np.errstate(divide="ignore", invalid="ignore"):
        r = X.T @ Y / den
        df = mask.sum() - np.linalg.matrix_rank(C) - 1
        t = r * np.sqrt(df / (1 - r ** 2))
        p = 2 * stats.t.sf(np.abs(t), df=df)

    cols = [event_pos[event] for event in events]
    PARTIAL_R[:, cols] = r.astype(np.float32)
    PARTIAL_P[:, cols] = p.astype(np.float32)


# FDR
valid = np.isfinite(PARTIAL_P)
PARTIAL_PADJ = np.full(PARTIAL_P.shape, np.nan, dtype=np.float32)
PARTIAL_PADJ[valid] = multipletests(PARTIAL_P[valid], method="fdr_bh")[1].astype(np.float32)


# 在Pearson + Spearman候选中继续筛选
robust_edges["partial_r"] = [
    PARTIAL_R[np.where(rbp_ids == g)[0][0], np.where(event_ids == e)[0][0]]
    for g, e in zip(robust_edges["gene_id"], robust_edges["EventID"])
]

robust_edges["partial_padj"] = [
    PARTIAL_PADJ[np.where(rbp_ids == g)[0][0], np.where(event_ids == e)[0][0]]
    for g, e in zip(robust_edges["gene_id"], robust_edges["EventID"])
]

final_edges = robust_edges[
    (robust_edges["partial_padj"] < FDR_THRESH) &
    (robust_edges["partial_r"].abs() >= ADJ_COR_THRESH) &
    (np.sign(robust_edges["partial_r"]) == np.sign(robust_edges["pearson_r"]))
].copy()


# 保存
final_edges.to_csv(
    OUT_DIR / "11_RBP_AS_adjusted_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Final edges     :", len(final_edges))
print("RBP regulators  :", final_edges["gene_id"].nunique())
print("AS events       :", final_edges["EventID"].nunique())
print("AS genes        :", final_edges["AS_gene"].nunique())

display(final_edges["direction"].value_counts())

print("\nTop RBP:")
display(final_edges["RBP_name"].value_counts().head(20))

In [ ]:
# %% Cell 16：校正sample和region后的RBP-AS关联

# 参数
SAMPLE_COR_THRESH = 0.25
FDR_THRESH = 0.05


# 处理数据
SAMPLE_R = np.full(R.shape, np.nan, dtype=np.float32)
SAMPLE_P = np.full(R.shape, np.nan, dtype=np.float32)

for pattern, events in event_regions.groupby(event_regions).groups.items():
    regions = pattern.split("|")
    mask = obs_info["region"].isin(regions).values

    X = expr_mat.iloc[mask].to_numpy(float)
    Y = psi_mat.loc[mask, list(events)].to_numpy(float)

    cov = pd.get_dummies(obs_info.loc[mask, ["region", "sample"]], drop_first=True, dtype=float)
    C = np.column_stack([np.ones(mask.sum()), cov.to_numpy()])
    Q, _ = np.linalg.qr(C)

    X = X - Q @ (Q.T @ X)
    Y = Y - Q @ (Q.T @ Y)

    den = np.sqrt((X ** 2).sum(axis=0)[:, None] * (Y ** 2).sum(axis=0)[None, :])

    with np.errstate(divide="ignore", invalid="ignore"):
        r = X.T @ Y / den
        df = mask.sum() - np.linalg.matrix_rank(C) - 1
        t = r * np.sqrt(df / (1 - r ** 2))
        p = 2 * stats.t.sf(np.abs(t), df=df)

    cols = [event_pos[event] for event in events]
    SAMPLE_R[:, cols] = r.astype(np.float32)
    SAMPLE_P[:, cols] = p.astype(np.float32)


# FDR
valid = np.isfinite(SAMPLE_P)
SAMPLE_PADJ = np.full(SAMPLE_P.shape, np.nan, dtype=np.float32)
SAMPLE_PADJ[valid] = multipletests(SAMPLE_P[valid], method="fdr_bh")[1].astype(np.float32)


# 加到Cell 15的候选边
rbp_index = pd.Series(np.arange(len(rbp_ids)), index=rbp_ids)
event_index = pd.Series(np.arange(len(event_ids)), index=event_ids)

ri = final_edges["gene_id"].map(rbp_index).to_numpy()
ej = final_edges["EventID"].map(event_index).to_numpy()

final_edges["sample_adj_r"] = SAMPLE_R[ri, ej]
final_edges["sample_adj_padj"] = SAMPLE_PADJ[ri, ej]

validated_edges = final_edges[
    (final_edges["sample_adj_padj"] < FDR_THRESH) &
    (final_edges["sample_adj_r"].abs() >= SAMPLE_COR_THRESH) &
    (np.sign(final_edges["sample_adj_r"]) == np.sign(final_edges["pearson_r"]))
].copy()


# 保存
validated_edges.to_csv(
    OUT_DIR / "12_RBP_AS_validated_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Validated edges :", len(validated_edges))
print("RBP regulators  :", validated_edges["gene_id"].nunique())
print("AS events       :", validated_edges["EventID"].nunique())
print("AS genes        :", validated_edges["AS_gene"].nunique())

display(validated_edges["direction"].value_counts())

print("\nTop RBP:")
display(validated_edges["RBP_name"].value_counts().head(20))

In [ ]:
# %% Cell 17：统计hub RBP

# 处理数据
hub_rbp = (
    validated_edges.groupby(["gene_id", "RBP_name", "RBP_class"])
    .agg(
        n_events=("EventID", "nunique"),
        n_genes=("AS_gene", "nunique"),
        mean_abs_r=("sample_adj_r", lambda x: x.abs().mean()),
    )
    .sort_values(["n_events", "mean_abs_r"], ascending=False)
    .reset_index()
)


# 保存
hub_rbp.to_csv(OUT_DIR / "13_RBP_hub_summary.tsv", sep="\t", index=False)


# 检查
display(hub_rbp.head(30))

In [ ]:
# %% Cell 18：AS event分类并去除self-edge

# 处理数据
sig_summary = (
    rmats_events[rmats_events["AS_sig"]]
    .groupby(["EventID", "comparison"])["region"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

sig_summary = sig_summary.rename(columns={
    "AD_vs_Young": "n_AD_regions",
    "Aged_vs_Young": "n_Aged_regions"
})

sig_summary["AS_class"] = np.select(
    [
        (sig_summary["n_AD_regions"] > 0) & (sig_summary["n_Aged_regions"] == 0),
        (sig_summary["n_AD_regions"] > 0) & (sig_summary["n_Aged_regions"] > 0),
        (sig_summary["n_AD_regions"] == 0) & (sig_summary["n_Aged_regions"] > 0),
    ],
    ["AD only", "Shared", "Aged only"],
    default="Other"
)

validated_edges = validated_edges.merge(sig_summary, on="EventID", how="left")

# 去除RBP自身基因的AS事件
validated_edges = validated_edges[validated_edges["RBP_name"] != validated_edges["AS_gene"]].copy()


# 保存
validated_edges.to_csv(
    OUT_DIR / "14_RBP_AS_validated_edges_annotated.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Edges       :", len(validated_edges))
print("RBP         :", validated_edges["RBP_name"].nunique())
print("AS events   :", validated_edges["EventID"].nunique())
print("AS genes    :", validated_edges["AS_gene"].nunique())

display(validated_edges["AS_class"].value_counts())

In [ ]:
# %% Cell 19：统计RBP连接的AS类型

# 处理数据
rbp_class_summary = (
    validated_edges
    .groupby(["RBP_name", "AS_class"])["EventID"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["AD only", "Shared", "Aged only"]:
    if col not in rbp_class_summary.columns:
        rbp_class_summary[col] = 0

rbp_class_summary["Total"] = (
    rbp_class_summary["AD only"] +
    rbp_class_summary["Shared"] +
    rbp_class_summary["Aged only"]
)

rbp_class_summary = rbp_class_summary.sort_values("Total", ascending=False)


# 保存
rbp_class_summary.to_csv(OUT_DIR / "15_RBP_AS_class_summary.tsv", sep="\t", index=False)


# 检查
display(rbp_class_summary.head(30))

In [ ]:
# %% Cell 20：筛选主网络候选

# 参数
MIN_EVENTS_PER_RBP = 3


# 处理数据
hub_rbp = (
    validated_edges.groupby(["gene_id", "RBP_name", "RBP_class"])
    .agg(
        n_events=("EventID", "nunique"),
        n_genes=("AS_gene", "nunique"),
        mean_abs_r=("sample_adj_r", lambda x: x.abs().mean())
    )
    .reset_index()
)

hub_rbp = hub_rbp[hub_rbp["n_events"] >= MIN_EVENTS_PER_RBP].copy()
network_edges = validated_edges[validated_edges["RBP_name"].isin(hub_rbp["RBP_name"])].copy()


# 保存
hub_rbp.to_csv(OUT_DIR / "16_hub_RBP.tsv", sep="\t", index=False)
network_edges.to_csv(OUT_DIR / "17_network_candidate_edges.tsv.gz", sep="\t", index=False, compression="gzip")


# 检查
print("Hub RBP       :", len(hub_rbp))
print("Network edges :", len(network_edges))
print("AS events     :", network_edges["EventID"].nunique())
print("AS genes      :", network_edges["AS_gene"].nunique())

display(
    hub_rbp.sort_values(["n_events", "mean_abs_r"], ascending=False).head(30)
)

In [ ]:
# %% Cell 21：RBP-AS网络模块划分

import networkx as nx

# 构建网络
G = nx.Graph()

for _, row in network_edges.iterrows():
    rbp_node = "RBP|" + row["RBP_name"]
    as_node = "AS|" + row["EventID"]

    G.add_node(rbp_node, node_type="RBP", label=row["RBP_name"])
    G.add_node(as_node, node_type="AS", label=row["AS_gene"], AS_class=row["AS_class"], AS_type=row["AS_type"])

    G.add_edge(
        rbp_node,
        as_node,
        weight=abs(row["sample_adj_r"]),
        correlation=row["sample_adj_r"]
    )

# module detection
communities = nx.community.greedy_modularity_communities(G, weight="weight")

module_map = {}
for i, community in enumerate(communities, 1):
    for node in community:
        module_map[node] = i

nx.set_node_attributes(G, module_map, "module")


# 检查
print("Nodes       :", G.number_of_nodes())
print("Edges       :", G.number_of_edges())
print("Modules     :", len(communities))

module_sizes = pd.Series({i + 1: len(x) for i, x in enumerate(communities)}, name="n_nodes")
display(module_sizes.head(20))

In [ ]:
# %% Cell 22：统计network modules

# 处理数据
module_rows = []

for node, data in G.nodes(data=True):
    module_rows.append({
        "node": node,
        "module": data["module"],
        "node_type": data["node_type"],
        "label": data["label"],
        "AS_class": data.get("AS_class", np.nan),
        "AS_type": data.get("AS_type", np.nan),
    })

module_nodes = pd.DataFrame(module_rows)

module_summary = (
    module_nodes.groupby("module")
    .agg(
        n_nodes=("node", "size"),
        n_RBP=("node_type", lambda x: (x == "RBP").sum()),
        n_AS=("node_type", lambda x: (x == "AS").sum()),
    )
    .reset_index()
)

as_module = module_nodes[module_nodes["node_type"] == "AS"]

class_count = (
    as_module.groupby(["module", "AS_class"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

module_summary = module_summary.merge(class_count, on="module", how="left")
module_summary = module_summary.sort_values("n_nodes", ascending=False)


# 保存
module_nodes.to_csv(OUT_DIR / "18_network_module_nodes.tsv", sep="\t", index=False)
module_summary.to_csv(OUT_DIR / "19_network_module_summary.tsv", sep="\t", index=False)


# 检查
display(module_summary.head(20))

In [ ]:
# %% Cell 23：查看各module核心RBP

# 参数
TOP_RBP = 10
TOP_MODULES = 10


# 处理数据
degree = dict(G.degree(weight="weight"))

rbp_modules = module_nodes[module_nodes["node_type"] == "RBP"].copy()
rbp_modules["weighted_degree"] = rbp_modules["node"].map(degree)

top_module_ids = module_summary.head(TOP_MODULES)["module"]

top_rbp_modules = (
    rbp_modules[rbp_modules["module"].isin(top_module_ids)]
    .sort_values(["module", "weighted_degree"], ascending=[True, False])
    .groupby("module")
    .head(TOP_RBP)
)


# 保存
top_rbp_modules.to_csv(OUT_DIR / "20_top_RBP_by_module.tsv", sep="\t", index=False)


# 检查
display(top_rbp_modules[["module", "label", "weighted_degree"]])

In [ ]:
# %% Cell 24：整理high-confidence RBP-AS edges

# 参数
FINAL_FDR = 0.01
MIN_PSI_NONZERO = 10


# PSI非零数量
psi_nonzero = (
    psi_model.groupby("EventID")["PSI"]
    .apply(lambda x: (x > 0).sum())
    .rename("n_nonzero_PSI")
)


# 同脑区AD/Aged显著情况
sig = rmats_events[rmats_events["AS_sig"]][["EventID", "region", "comparison"]].drop_duplicates()

status = (
    sig.assign(value=1)
    .pivot_table(index=["EventID", "region"], columns="comparison", values="value", fill_value=0)
    .reset_index()
)

for col in ["AD_vs_Young", "Aged_vs_Young"]:
    if col not in status.columns:
        status[col] = 0

status["shared_region"] = (status["AD_vs_Young"] > 0) & (status["Aged_vs_Young"] > 0)

event_class = (
    status.groupby("EventID")
    .agg(
        n_AD_regions=("AD_vs_Young", "sum"),
        n_Aged_regions=("Aged_vs_Young", "sum"),
        n_shared_regions=("shared_region", "sum"),
    )
    .reset_index()
)

event_class["AS_class"] = np.select(
    [
        event_class["n_shared_regions"] > 0,
        (event_class["n_AD_regions"] > 0) & (event_class["n_Aged_regions"] == 0),
        (event_class["n_AD_regions"] == 0) & (event_class["n_Aged_regions"] > 0),
    ],
    ["Shared", "AD only", "Aged only"],
    default="Both region-specific"
)


# 最终严格网络
drop_cols = ["n_AD_regions", "n_Aged_regions", "n_shared_regions", "AS_class"]
network_final = validated_edges.drop(columns=[x for x in drop_cols if x in validated_edges.columns])

network_final = network_final.merge(psi_nonzero, on="EventID").merge(event_class, on="EventID")

network_final = network_final[
    (network_final["n_nonzero_PSI"] >= MIN_PSI_NONZERO) &
    (network_final["partial_padj"] < FINAL_FDR) &
    (network_final["sample_adj_padj"] < FINAL_FDR)
].copy()


# 保存
network_final.to_csv(
    OUT_DIR / "21_high_confidence_RBP_AS_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Edges      :", len(network_final))
print("RBP        :", network_final["RBP_name"].nunique())
print("AS events  :", network_final["EventID"].nunique())
print("AS genes   :", network_final["AS_gene"].nunique())

display(network_final["AS_class"].value_counts())

In [ ]:
# %% Cell 25：重新进行network module detection

import networkx as nx

# 构建网络
G = nx.Graph()

for row in network_final.itertuples():
    rbp_node = "RBP|" + row.RBP_name
    as_node = "AS|" + row.EventID

    G.add_node(rbp_node, node_type="RBP", label=row.RBP_name)
    G.add_node(as_node, node_type="AS", label=row.AS_gene, AS_class=row.AS_class, AS_type=row.AS_type)
    G.add_edge(rbp_node, as_node, weight=abs(row.sample_adj_r), correlation=row.sample_adj_r)


# module detection
communities = nx.community.greedy_modularity_communities(G, weight="weight")

module_map = {}
for i, community in enumerate(communities, 1):
    for node in community:
        module_map[node] = i

nx.set_node_attributes(G, module_map, "module")


# 整理结果
module_nodes = pd.DataFrame([
    {
        "node": node,
        "module": data["module"],
        "node_type": data["node_type"],
        "label": data["label"],
        "AS_class": data.get("AS_class", np.nan),
        "AS_type": data.get("AS_type", np.nan),
    }
    for node, data in G.nodes(data=True)
])

module_summary = (
    module_nodes.groupby("module")
    .agg(
        n_nodes=("node", "size"),
        n_RBP=("node_type", lambda x: (x == "RBP").sum()),
        n_AS=("node_type", lambda x: (x == "AS").sum()),
    )
    .reset_index()
    .sort_values("n_nodes", ascending=False)
)


# 保存
module_nodes.to_csv(OUT_DIR / "22_high_confidence_module_nodes.tsv", sep="\t", index=False)
module_summary.to_csv(OUT_DIR / "23_high_confidence_module_summary.tsv", sep="\t", index=False)


# 检查
print("Nodes   :", G.number_of_nodes())
print("Edges   :", G.number_of_edges())
print("Modules :", len(communities))

display(module_summary.head(15))

In [ ]:
# %% Cell 26：整理主要network modules

# 参数
TOP_MODULES = 20
TOP_NAMES = 10


# 处理数据
rows = []

for module in module_summary.head(TOP_MODULES)["module"]:
    nodes = module_nodes[module_nodes["module"] == module]
    sub = G.subgraph(nodes["node"])

    rbp_nodes = nodes.loc[nodes["node_type"] == "RBP", "node"]
    as_nodes = nodes.loc[nodes["node_type"] == "AS", "node"]

    rbp_rank = sorted(rbp_nodes, key=lambda x: sub.degree(x, weight="weight"), reverse=True)[:TOP_NAMES]
    as_rank = sorted(as_nodes, key=lambda x: sub.degree(x, weight="weight"), reverse=True)[:TOP_NAMES]

    classes = nodes.loc[nodes["node_type"] == "AS", "AS_class"].value_counts()

    rows.append({
        "module": module,
        "n_nodes": len(nodes),
        "n_RBP": len(rbp_nodes),
        "n_AS": len(as_nodes),
        "n_edges": sub.number_of_edges(),
        "Shared": classes.get("Shared", 0),
        "AD_only": classes.get("AD only", 0),
        "Aged_only": classes.get("Aged only", 0),
        "Both_region_specific": classes.get("Both region-specific", 0),
        "top_RBP": ", ".join(G.nodes[x]["label"] for x in rbp_rank),
        "top_AS": ", ".join(f'{G.nodes[x]["label"]}-{G.nodes[x]["AS_type"]}' for x in as_rank),
    })

module_review = pd.DataFrame(rows)


# 保存
module_review.to_csv(OUT_DIR / "24_module_review.tsv", sep="\t", index=False)


# 检查
display(module_review)

In [ ]:
# %% Cell 27：整理module内部具体网络边

# 处理数据
node_module = dict(zip(module_nodes["node"], module_nodes["module"]))

module_edges = network_final.copy()
module_edges["RBP_node"] = "RBP|" + module_edges["RBP_name"]
module_edges["AS_node"] = "AS|" + module_edges["EventID"]

module_edges["RBP_module"] = module_edges["RBP_node"].map(node_module)
module_edges["AS_module"] = module_edges["AS_node"].map(node_module)

module_edges = module_edges[
    module_edges["RBP_module"] == module_edges["AS_module"]
].copy()

module_edges["module"] = module_edges["RBP_module"].astype(int)
module_edges["abs_r"] = module_edges["sample_adj_r"].abs()

module_edges = module_edges.sort_values(
    ["module", "abs_r"],
    ascending=[True, False]
)


# 保存
module_edges.to_csv(
    OUT_DIR / "25_module_internal_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
print("Internal edges :", len(module_edges))
print("Modules        :", module_edges["module"].nunique())

display(
    module_edges[
        ["module", "RBP_name", "AS_gene", "AS_type", "AS_class", "sample_adj_r"]
    ].head(30)
)

# Cell 28：全局 high-confidence network overview

In [ ]:
# %% Cell 28：全局RBP-AS network overview

import matplotlib.pyplot as plt

# 参数
figsize = (6, 6)
node_size_rbp = 8
node_size_as = 18
edge_alpha = 0.18

AS_COLORS = {
    "AD only": "#DD8452",
    "Aged only": "#4C72B0",
    "Shared": "#8F8F8F",
    "Both region-specific": "#9467BD",
}


# 布局
pos = nx.spring_layout(G, seed=10, weight="weight", iterations=100)

rbp_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "RBP"]
as_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "AS"]


# 画图
fig, ax = plt.subplots(figsize=figsize)

nx.draw_networkx_edges(
    G, pos, ax=ax,
    width=[0.4 + 1.2 * G[u][v]["weight"] for u, v in G.edges()],
    edge_color="#BDBDBD",
    alpha=edge_alpha
)

nx.draw_networkx_nodes(
    G, pos, nodelist=rbp_nodes, ax=ax,
    node_size=node_size_rbp,
    node_color="#BDBDBD",
    linewidths=0
)

for group, color in AS_COLORS.items():
    nodes = [n for n in as_nodes if G.nodes[n]["AS_class"] == group]
    nx.draw_networkx_nodes(
        G, pos, nodelist=nodes, ax=ax,
        node_size=node_size_as,
        node_color=color,
        linewidths=0
    )

ax.axis("off")
plt.tight_layout()

plt.savefig(
    OUT_DIR / "Fig_global_RBP_AS_network.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# %% Cell 29：Module 7 regulatory network

import matplotlib.pyplot as plt
import networkx as nx

# 参数
MODULE = 7
figsize = (6, 5.5)
fontsize = 9

AS_COLORS = {
    "AD only": "#DD8452",
    "Aged only": "#4C72B0",
    "Shared": "#8F8F8F",
    "Both region-specific": "#9467BD",
}

plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# 读取当前module
df = module_edges[module_edges["module"] == MODULE].copy()

rbps = df["RBP_name"].drop_duplicates().tolist()
events = df[["EventID", "AS_gene", "AS_type", "AS_class"]].drop_duplicates()
event_ids = events["EventID"].tolist()

Gm = nx.Graph()
Gm.add_nodes_from(rbps, node_type="RBP")

for row in events.itertuples():
    Gm.add_node(row.EventID, node_type="AS", label=f"{row.AS_gene}-{row.AS_type}", AS_class=row.AS_class)

for row in df.itertuples():
    Gm.add_edge(row.RBP_name, row.EventID, weight=abs(row.sample_adj_r), correlation=row.sample_adj_r)


# 布局
pos = nx.bipartite_layout(Gm, rbps, align="vertical", scale=1.8)

# 画图
fig, ax = plt.subplots(figsize=figsize)

pos_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] > 0]
neg_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] < 0]

nx.draw_networkx_edges(Gm, pos, edgelist=pos_edges, ax=ax, edge_color="#666666",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in pos_edges], alpha=0.65)

nx.draw_networkx_edges(Gm, pos, edgelist=neg_edges, ax=ax, edge_color="#B0B0B0",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in neg_edges], style="dashed", alpha=0.65)

nx.draw_networkx_nodes(Gm, pos, nodelist=rbps, ax=ax, node_size=45,
                       node_color="#5F5F5F", linewidths=0)

for group, color in AS_COLORS.items():
    nodes = events.loc[events["AS_class"] == group, "EventID"].tolist()
    nx.draw_networkx_nodes(Gm, pos, nodelist=nodes, ax=ax, node_size=110,
                           node_color=color, linewidths=0)

# 标签
for node in rbps:
    x, y = pos[node]
    ax.text(x - 0.06, y, node, ha="right", va="center", fontsize=fontsize, fontstyle="italic")

for node in event_ids:
    x, y = pos[node]
    ax.text(x + 0.06, y, Gm.nodes[node]["label"], ha="left", va="center",
            fontsize=fontsize, fontstyle="italic")

ax.text(-1.8, 1.95, "RBP", ha="center", fontsize=fontsize + 1)
ax.text(1.8, 1.95, "AS event", ha="center", fontsize=fontsize + 1)

ax.axis("off")
plt.tight_layout()

plt.savefig(OUT_DIR / f"Module{MODULE}_RBP_AS_network.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# %% Cell 29：Module 7 regulatory network

import matplotlib.pyplot as plt
import networkx as nx

# 参数
MODULE = 6
figsize = (6, 5.5)
fontsize = 9

AS_COLORS = {
    "AD only": "#DD8452",
    "Aged only": "#4C72B0",
    "Shared": "#8F8F8F",
    "Both region-specific": "#9467BD",
}

plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# 读取当前module
df = module_edges[module_edges["module"] == MODULE].copy()

rbps = df["RBP_name"].drop_duplicates().tolist()
events = df[["EventID", "AS_gene", "AS_type", "AS_class"]].drop_duplicates()
event_ids = events["EventID"].tolist()

Gm = nx.Graph()
Gm.add_nodes_from(rbps, node_type="RBP")

for row in events.itertuples():
    Gm.add_node(row.EventID, node_type="AS", label=f"{row.AS_gene}-{row.AS_type}", AS_class=row.AS_class)

for row in df.itertuples():
    Gm.add_edge(row.RBP_name, row.EventID, weight=abs(row.sample_adj_r), correlation=row.sample_adj_r)


# 布局
pos = nx.bipartite_layout(Gm, rbps, align="vertical", scale=1.8)

# 画图
fig, ax = plt.subplots(figsize=figsize)

pos_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] > 0]
neg_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] < 0]

nx.draw_networkx_edges(Gm, pos, edgelist=pos_edges, ax=ax, edge_color="#666666",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in pos_edges], alpha=0.65)

nx.draw_networkx_edges(Gm, pos, edgelist=neg_edges, ax=ax, edge_color="#B0B0B0",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in neg_edges], style="dashed", alpha=0.65)

nx.draw_networkx_nodes(Gm, pos, nodelist=rbps, ax=ax, node_size=45,
                       node_color="#5F5F5F", linewidths=0)

for group, color in AS_COLORS.items():
    nodes = events.loc[events["AS_class"] == group, "EventID"].tolist()
    nx.draw_networkx_nodes(Gm, pos, nodelist=nodes, ax=ax, node_size=110,
                           node_color=color, linewidths=0)

# 标签
for node in rbps:
    x, y = pos[node]
    ax.text(x - 0.06, y, node, ha="right", va="center", fontsize=fontsize, fontstyle="italic")

for node in event_ids:
    x, y = pos[node]
    ax.text(x + 0.06, y, Gm.nodes[node]["label"], ha="left", va="center",
            fontsize=fontsize, fontstyle="italic")

ax.text(-1.8, 1.95, "RBP", ha="center", fontsize=fontsize + 1)
ax.text(1.8, 1.95, "AS event", ha="center", fontsize=fontsize + 1)

ax.axis("off")
plt.tight_layout()

plt.savefig(OUT_DIR / f"Module{MODULE}_RBP_AS_network.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# %% Cell 29：Module 7 regulatory network

import matplotlib.pyplot as plt
import networkx as nx

# 参数
MODULE = 5
figsize = (6, 5.5)
fontsize = 9

AS_COLORS = {
    "AD only": "#DD8452",
    "Aged only": "#4C72B0",
    "Shared": "#8F8F8F",
    "Both region-specific": "#9467BD",
}

plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# 读取当前module
df = module_edges[module_edges["module"] == MODULE].copy()

rbps = df["RBP_name"].drop_duplicates().tolist()
events = df[["EventID", "AS_gene", "AS_type", "AS_class"]].drop_duplicates()
event_ids = events["EventID"].tolist()

Gm = nx.Graph()
Gm.add_nodes_from(rbps, node_type="RBP")

for row in events.itertuples():
    Gm.add_node(row.EventID, node_type="AS", label=f"{row.AS_gene}-{row.AS_type}", AS_class=row.AS_class)

for row in df.itertuples():
    Gm.add_edge(row.RBP_name, row.EventID, weight=abs(row.sample_adj_r), correlation=row.sample_adj_r)


# 布局
pos = nx.bipartite_layout(Gm, rbps, align="vertical", scale=1.8)

# 画图
fig, ax = plt.subplots(figsize=figsize)

pos_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] > 0]
neg_edges = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] < 0]

nx.draw_networkx_edges(Gm, pos, edgelist=pos_edges, ax=ax, edge_color="#666666",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in pos_edges], alpha=0.65)

nx.draw_networkx_edges(Gm, pos, edgelist=neg_edges, ax=ax, edge_color="#B0B0B0",
                       width=[1.5 * Gm[u][v]["weight"] for u, v in neg_edges], style="dashed", alpha=0.65)

nx.draw_networkx_nodes(Gm, pos, nodelist=rbps, ax=ax, node_size=45,
                       node_color="#5F5F5F", linewidths=0)

for group, color in AS_COLORS.items():
    nodes = events.loc[events["AS_class"] == group, "EventID"].tolist()
    nx.draw_networkx_nodes(Gm, pos, nodelist=nodes, ax=ax, node_size=110,
                           node_color=color, linewidths=0)

# 标签
for node in rbps:
    x, y = pos[node]
    ax.text(x - 0.06, y, node, ha="right", va="center", fontsize=fontsize, fontstyle="italic")

for node in event_ids:
    x, y = pos[node]
    ax.text(x + 0.06, y, Gm.nodes[node]["label"], ha="left", va="center",
            fontsize=fontsize, fontstyle="italic")

ax.text(-1.8, 1.95, "RBP", ha="center", fontsize=fontsize + 1)
ax.text(1.8, 1.95, "AS event", ha="center", fontsize=fontsize + 1)

ax.axis("off")
plt.tight_layout()

plt.savefig(OUT_DIR / f"Module{MODULE}_RBP_AS_network.pdf", bbox_inches="tight")
plt.show()

# Cell 29：给 network AS events 加 Gene–AS 分类

In [ ]:
# %% Cell 29：给network AS events加入Gene-AS分类

# 参数
GENE_FILE = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_DEGs/"
    "spot_level_DE_outer_min10/spot_DEG_all_regions_all_comparisons.tsv.gz"
)
COMPARISONS = ["Aged_vs_Young", "AD_vs_Young"]

GENE_STABLE_FC = 0.50
GENE_DE_FC = 0.75
GENE_PADJ = 0.01


# 读取数据
gene = pd.read_csv(GENE_FILE, sep="\t")
gene = gene[gene["comparison"].isin(COMPARISONS)].copy()
gene["gene_name"] = gene["Symbol"].astype(str).str.strip()

gene = gene.sort_values("padj", na_position="last").drop_duplicates(
    ["region", "comparison", "gene_name"]
)


# 处理数据
gene["gene_class"] = "Intermediate"
gene.loc[gene["log2FC"].abs() <= GENE_STABLE_FC, "gene_class"] = "Gene stable"
gene.loc[
    (gene["padj"] < GENE_PADJ) & (gene["log2FC"].abs() >= GENE_DE_FC),
    "gene_class"
] = "Gene DE"

event_gene = (
    rmats_events[rmats_events["AS_sig"]][["EventID", "region", "comparison", "gene_name"]]
    .merge(
        gene[["region", "comparison", "gene_name", "log2FC", "padj", "gene_class"]],
        on=["region", "comparison", "gene_name"],
        how="left"
    )
)

event_gene["Gene_AS_class"] = event_gene["gene_class"].map({
    "Gene stable": "AS only",
    "Gene DE": "DEG + AS",
    "Intermediate": "Intermediate + AS",
})

event_class_gene = (
    event_gene.groupby("EventID")["Gene_AS_class"]
    .agg(lambda x: x.dropna().unique())
    .reset_index()
)

event_class_gene["Gene_AS_class"] = event_class_gene["Gene_AS_class"].apply(
    lambda x: x[0] if len(x) == 1 else "Mixed"
)

network_final = network_final.drop(columns=["Gene_AS_class"], errors="ignore")
network_final = network_final.merge(event_class_gene, on="EventID", how="left")


# 保存
network_final.to_csv(
    OUT_DIR / "26_high_confidence_RBP_AS_edges_with_gene_class.tsv.gz",
    sep="\t", index=False, compression="gzip"
)


# 检查
event_summary = network_final[["EventID", "AS_gene", "Gene_AS_class"]].drop_duplicates()

display(event_summary["Gene_AS_class"].value_counts())
display(event_summary["Gene_AS_class"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# %% Cell 30：统计各module的Gene-AS分类

# 处理数据
module_gene_class = (
    module_edges[["module", "EventID"]]
    .drop_duplicates()
    .merge(event_class_gene, on="EventID", how="left")
)

module_gene_summary = (
    module_gene_class.groupby(["module", "Gene_AS_class"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["AS only", "Mixed", "Intermediate + AS", "DEG + AS"]:
    if col not in module_gene_summary.columns:
        module_gene_summary[col] = 0

module_gene_summary["Total"] = module_gene_summary[
    ["AS only", "Mixed", "Intermediate + AS", "DEG + AS"]
].sum(axis=1)

module_gene_summary["AS_only_pct"] = (
    module_gene_summary["AS only"] / module_gene_summary["Total"] * 100
)

module_gene_summary = module_gene_summary.sort_values(
    ["AS only", "AS_only_pct"], ascending=False
)


# 保存
module_gene_summary.to_csv(
    OUT_DIR / "27_module_Gene_AS_summary.tsv",
    sep="\t", index=False
)


# 检查
display(module_gene_summary.head(20))

In [ ]:
# %% Cell 31：给module edges加入Gene-AS分类

# 处理数据
module_edges_anno = module_edges.drop(columns=["Gene_AS_class"], errors="ignore")
module_edges_anno = module_edges_anno.merge(
    event_class_gene,
    on="EventID",
    how="left"
)

# 保存
module_edges_anno.to_csv(
    OUT_DIR / "28_module_edges_with_Gene_AS_class.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

# 检查
display(
    module_edges_anno[
        ["module", "RBP_name", "AS_gene", "AS_type", "AS_class", "Gene_AS_class", "sample_adj_r"]
    ].head(30)
)

# 最终版

In [ ]:
# %% Cell 32：过滤污染基因，保留AS-only network

# 参数
EXCLUDE_GENES = [
    "Cox8b", "Actc1", "Acta1", "Myl7", "Tnni3",
    "Ckmt2", "Ckm", "Csrp3", "Hrc", "Tcap",
    "Klhl40", "Sln", "Pgam2",
    "Krt10", "Kap", "Aldob", "Spink1", "Htra4",
    "Miox", "Napsa", "Gkn3", "Car6", "Crisp3",
    "Pip", "Defb11", "Mucl2", "Scgb1b27", "Lct", "Tsacc",
    "Klk1", "Klk1b1", "Klk1b3", "Klk1b4", "Klk1b5",
    "Klk1b8", "Klk1b9", "Klk1b11", "Klk1b16",
    "Klk1b21", "Klk1b22", "Klk1b26", "Klk1b27",
    "Scgb2b26", "Scgb2b27", "Cd3d", "Cd3g", "Cox7a1",
    "Fxyd2", "Gm15543", "Ly6d", "S100g", "Trbc1",
    "Wfdc12", "Fabp4", "Fbp1",
    "Gh", "Guca2b", "Nppb", "Nppa", "Tnnc1",
    "Wfdc15b", "Dmkn", "Krt15",
    "Trbc2", "Wfdc18", "Cox6a2", "Krt8", "Krt18",
]

# 处理数据
network_clean = network_final.copy()
network_clean = network_clean[
    ~network_clean["AS_gene"].isin(EXCLUDE_GENES) &
    ~network_clean["RBP_name"].isin(EXCLUDE_GENES)
].copy()

network_as_only = network_clean[
    network_clean["Gene_AS_class"] == "AS only"
].copy()

# 保存
network_clean.to_csv(
    OUT_DIR / "29_high_confidence_RBP_AS_edges_clean.tsv.gz",
    sep="\t", index=False, compression="gzip"
)
network_as_only.to_csv(
    OUT_DIR / "30_AS_only_RBP_AS_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

# 检查
print("Clean network")
print("Edges     :", len(network_clean))
print("RBP       :", network_clean["RBP_name"].nunique())
print("AS events :", network_clean["EventID"].nunique())
print("AS genes  :", network_clean["AS_gene"].nunique())

print("\nAS-only network")
print("Edges     :", len(network_as_only))
print("RBP       :", network_as_only["RBP_name"].nunique())
print("AS events :", network_as_only["EventID"].nunique())
print("AS genes  :", network_as_only["AS_gene"].nunique())

display(network_as_only["AS_class"].value_counts())

In [ ]:
# %% Cell 33：AS-only network module detection

import networkx as nx

# 构建网络
G_as = nx.Graph()

for row in network_as_only.itertuples():
    rbp_node = "RBP|" + row.RBP_name
    as_node = "AS|" + row.EventID

    G_as.add_node(rbp_node, node_type="RBP", label=row.RBP_name)
    G_as.add_node(
        as_node,
        node_type="AS",
        label=row.AS_gene,
        AS_type=row.AS_type,
        AS_class=row.AS_class,
        Gene_AS_class=row.Gene_AS_class
    )
    G_as.add_edge(
        rbp_node, as_node,
        weight=abs(row.sample_adj_r),
        correlation=row.sample_adj_r
    )

# module detection
communities_as = nx.community.greedy_modularity_communities(G_as, weight="weight")

module_map_as = {}
for i, community in enumerate(communities_as, 1):
    for node in community:
        module_map_as[node] = i

nx.set_node_attributes(G_as, module_map_as, "module")

# 整理节点表
module_nodes_as = pd.DataFrame([
    {
        "node": node,
        "module": data["module"],
        "node_type": data["node_type"],
        "label": data["label"],
        "AS_type": data.get("AS_type", np.nan),
        "AS_class": data.get("AS_class", np.nan),
        "Gene_AS_class": data.get("Gene_AS_class", np.nan),
    }
    for node, data in G_as.nodes(data=True)
])

module_summary_as = (
    module_nodes_as.groupby("module")
    .agg(
        n_nodes=("node", "size"),
        n_RBP=("node_type", lambda x: (x == "RBP").sum()),
        n_AS=("node_type", lambda x: (x == "AS").sum()),
    )
    .reset_index()
    .sort_values("n_nodes", ascending=False)
)

# 保存
module_nodes_as.to_csv(OUT_DIR / "31_AS_only_module_nodes.tsv", sep="\t", index=False)
module_summary_as.to_csv(OUT_DIR / "32_AS_only_module_summary.tsv", sep="\t", index=False)

# 检查
print("Nodes   :", G_as.number_of_nodes())
print("Edges   :", G_as.number_of_edges())
print("Modules :", len(communities_as))
display(module_summary_as.head(20))

In [ ]:
# %% Cell 34：整理AS-only主要module信息

# 参数
TOP_MODULES = 20
TOP_NAMES = 10

# 处理数据
rows = []

for module in module_summary_as.head(TOP_MODULES)["module"]:
    nodes = module_nodes_as[module_nodes_as["module"] == module]
    sub = G_as.subgraph(nodes["node"])

    rbp_nodes = nodes.loc[nodes["node_type"] == "RBP", "node"]
    as_nodes = nodes.loc[nodes["node_type"] == "AS", "node"]

    rbp_rank = sorted(rbp_nodes, key=lambda x: sub.degree(x, weight="weight"), reverse=True)[:TOP_NAMES]
    as_rank = sorted(as_nodes, key=lambda x: sub.degree(x, weight="weight"), reverse=True)[:TOP_NAMES]

    as_info = (
        network_as_only[network_as_only["EventID"].isin([x.replace("AS|", "") for x in as_nodes])]
        [["EventID", "AS_gene", "AS_type", "AS_class"]]
        .drop_duplicates()
        .set_index("EventID")
    )

    top_as_labels = []
    for x in as_rank:
        eid = x.replace("AS|", "")
        top_as_labels.append(f'{as_info.loc[eid, "AS_gene"]}-{as_info.loc[eid, "AS_type"]}')

    rows.append({
        "module": module,
        "n_nodes": len(nodes),
        "n_RBP": len(rbp_nodes),
        "n_AS": len(as_nodes),
        "n_edges": sub.number_of_edges(),
        "top_RBP": ", ".join(G_as.nodes[x]["label"] for x in rbp_rank),
        "top_AS": ", ".join(top_as_labels),
    })

module_review_as = pd.DataFrame(rows)

# 保存
module_review_as.to_csv(OUT_DIR / "33_AS_only_module_review.tsv", sep="\t", index=False)

# 检查
display(module_review_as)

In [ ]:
# %% Cell 35：整理AS-only module内部边

# 处理数据
node_module_as = dict(zip(module_nodes_as["node"], module_nodes_as["module"]))

module_edges_as = network_as_only.copy()
module_edges_as["RBP_node"] = "RBP|" + module_edges_as["RBP_name"]
module_edges_as["AS_node"] = "AS|" + module_edges_as["EventID"]

module_edges_as["RBP_module"] = module_edges_as["RBP_node"].map(node_module_as)
module_edges_as["AS_module"] = module_edges_as["AS_node"].map(node_module_as)

module_edges_as = module_edges_as[
    module_edges_as["RBP_module"] == module_edges_as["AS_module"]
].copy()

module_edges_as["module"] = module_edges_as["RBP_module"].astype(int)
module_edges_as["abs_r"] = module_edges_as["sample_adj_r"].abs()
module_edges_as = module_edges_as.sort_values(["module", "abs_r"], ascending=[True, False])

# 保存
module_edges_as.to_csv(
    OUT_DIR / "34_AS_only_module_internal_edges.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

# 检查
print("Internal edges :", len(module_edges_as))
print("Modules        :", module_edges_as["module"].nunique())

display(
    module_edges_as[
        ["module", "RBP_name", "AS_gene", "AS_type", "AS_class", "sample_adj_r"]
    ].head(30)
)

In [ ]:
# %% Cell 36：绘制AS-only regulatory module

import matplotlib.pyplot as plt
import networkx as nx

# 参数
MODULE = 3
TOP_PER_EVENT = 3
figsize = (6, 5)
fontsize = 9

AS_COLORS = {
    "AD only": "#DD8452",
    "Aged only": "#4C72B0",
    "Shared": "#8F8F8F",
    "Both region-specific": "#9467BD",
}

plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# 处理数据
df = module_edges_as[module_edges_as["module"] == MODULE].copy()
df_plot = df.sort_values("abs_r", ascending=False).groupby("EventID").head(TOP_PER_EVENT).copy()

event_info = df_plot[["EventID", "AS_gene", "AS_type", "AS_class"]].drop_duplicates()
event_order = df_plot.groupby("EventID").size().sort_values(ascending=False).index.tolist()
event_rank = {x: i for i, x in enumerate(event_order)}

rbp_order = (
    df_plot.assign(rank=df_plot["EventID"].map(event_rank))
    .groupby("RBP_name")["rank"].mean()
    .sort_values()
    .index.tolist()
)

Gm = nx.DiGraph()
for row in df_plot.itertuples():
    Gm.add_edge(row.RBP_name, row.EventID, weight=abs(row.sample_adj_r), correlation=row.sample_adj_r)

# 左右布局
pos = {}
for y, gene in zip(np.linspace(1, 0, len(rbp_order)), rbp_order):
    pos[gene] = (0, y)

for y, event in zip(np.linspace(1, 0, len(event_order)), event_order):
    pos[event] = (1, y)


# 画图
fig, ax = plt.subplots(figsize=figsize)

positive = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] > 0]
negative = [(u, v) for u, v, d in Gm.edges(data=True) if d["correlation"] < 0]

nx.draw_networkx_edges(
    Gm, pos, edgelist=positive, ax=ax,
    width=[1.8 * Gm[u][v]["weight"] for u, v in positive],
    edge_color="#555555", arrows=True, arrowsize=8, alpha=0.7
)

nx.draw_networkx_edges(
    Gm, pos, edgelist=negative, ax=ax,
    width=[1.8 * Gm[u][v]["weight"] for u, v in negative],
    edge_color="#AAAAAA", style="dashed", arrows=True, arrowsize=8, alpha=0.7
)

nx.draw_networkx_nodes(
    Gm, pos, nodelist=rbp_order, ax=ax,
    node_size=55, node_color="#555555", linewidths=0
)

for group, color in AS_COLORS.items():
    nodes = event_info.loc[event_info["AS_class"] == group, "EventID"].tolist()
    nx.draw_networkx_nodes(
        Gm, pos, nodelist=nodes, ax=ax,
        node_size=140, node_color=color, node_shape="s", linewidths=0
    )

# 标签
for gene in rbp_order:
    x, y = pos[gene]
    ax.text(x - 0.035, y, gene, ha="right", va="center", fontsize=fontsize, fontstyle="italic")

labels = event_info.set_index("EventID")
for event in event_order:
    x, y = pos[event]
    label = f'{labels.loc[event, "AS_gene"]}-{labels.loc[event, "AS_type"]}'
    ax.text(x + 0.035, y, label, ha="left", va="center", fontsize=fontsize, fontstyle="italic")

ax.text(0, 1.08, "RBP", ha="center", fontsize=fontsize + 1)
ax.text(1, 1.08, "Gene-stable + AS event", ha="center", fontsize=fontsize + 1)

ax.set_xlim(-0.45, 1.55)
ax.set_ylim(-0.08, 1.13)
ax.grid(False)
ax.axis("off")

plt.savefig(OUT_DIR / f"AS_only_Module{MODULE}_network.pdf", bbox_inches="tight")
plt.show()

# 000